# nestdb minimal notebook

offline by construction: the potion table rides inside the wheel, the
corpus is one local file, no network calls anywhere. 

setup: `pip install "nestdb[embed]" jupyter`

In [ ]:
import nest
from nest.embed_potion import potion_embedder

emb = potion_embedder()
print(emb.embedding_model, emb.embedding_dim)

In [ ]:
# build a tiny corpus once (reproducible: same bytes on every machine)
from pathlib import Path

docs = [
    "nest is a single-file vector database that works fully offline.",
    "every search hit carries a content-addressable citation.",
    "the potion static table embeds queries without a gpu or network.",
]
path = Path("demo_notebook.nest")
if not path.exists():
    chunks = [
        {
            "canonical_text": t,
            "source_uri": f"example://notebook/{i}",
            "byte_start": 0,
            "byte_end": len(t.encode()),
            "embedding": emb.embed_texts([t])[0],
        }
        for i, t in enumerate(docs)
    ]
    nest.build(
        str(path), emb.embedding_model, emb.embedding_dim,
        "notebook-example-1", emb.model_hash(), chunks, reproducible=True,
    )
db = nest.open(str(path))
db.validate()
print(db.file_hash)

In [ ]:
qvec = emb.embed_texts(["cited offline search"])[0]
for hit in db.retrieve(qvec, 2):
    print(f"{hit.score:.4f}  {hit.text}\n         {hit.citation_id}")